#### C. Implementation of the RSA function and its inverse

In [63]:
import os                                                            # For navegation in the file folders.
import secrets                                                       # To use entropy for true randomness.
from aux_funcs import *                                               # Auxiliar functions for the project.
from cryptography.hazmat.primitives.asymmetric import rsa            # For generating keys.
from cryptography.hazmat.primitives import hashes                    # Function H(n, r).

We start by creating the functions that will mask and unmask our private key. For the encryption, we use the formula:

**Enc(public_key, r_int) = r_int ^ e (mod n)**, where e is the exponent and n is the public number.



And, for the inverse function that decrypts, we use the formula:

**Dec(private_key, c_r) = c_r ^ d (mod n)**, where d is the number such that e * d (mod v) = 1

In [64]:
def rsa_trapdoor_permutation(public_key, r_int):                     # Pure RSA.
    e = public_key.public_numbers().e
    n = public_key.public_numbers().n
    return pow(r_int, e, n)

def rsa_inverse_trapdoor(private_key, c_r):                          # Inverse function of the one above.
    d = private_key.private_numbers().d
    n = private_key.private_numbers().public_numbers.n
    return pow(c_r, d, n)

Having the RSA functions ready, we can now define a way to encrypt and decrypt a message. The next function receives a public key and a message in its original state and transforms it into ciphertext:

1. Start by getting the number n from our public key. 
2. Randomize a number r to be encrypted, which must be smaller than n so that, when reversing (mod n), we know there is only one possible value for r.
3. Apply the trapdoor permutation to encrypt.
4. Transform r from bits to bytes, so that we can apply SHA 256 on it.
5. Divide message into blocks of 32 bytes.
6. Hash each block, measuring the hashing time.
6. Add the ciphertext block to our group of blocks, applying an XOR with the original block.

In [65]:
def custom_encrypt_decrypt(arguments):
    private_key, public_key, message, mode = arguments[0], arguments[1], arguments[2], arguments[3] 
    l_size = 32                                                          # SHA 256 output size
    ciphertext_blocks = []    
    if (mode == "Encrypting"):  
        enc_message = message
        n = public_key.public_numbers().n                                # STEP 1.: Get public number n to generate r
        r = secrets.randbelow(n)                                         # STEP 2.: Randomize a number r to be encrypted
        rsa_r = rsa_trapdoor_permutation(public_key, r)                  # STEP 3.: Apply the trapdoor permutation to r
        rsa_r_bytes = rsa_r.to_bytes(256, 'big')                         # STEP 4.1.: Transform encrypted r from bits into bytes (2048 bits = 256 bytes)
        ciphertext_blocks += [rsa_r_bytes]     
        r_bytes = r.to_bytes(256, 'big')                                 # STEP 4.2.: Transform original r from bits into bytes, to use in hashing operation
        hash_time = 0

    elif (mode == "Decrypting"):
        int_r = rsa_inverse_trapdoor(private_key, int.from_bytes(message[:256], 'big'))
        r_bytes = int_r.to_bytes(256, 'big')
        enc_message = message[256:]

    for i in range(0, len(enc_message), l_size):                
        block_index = i // l_size
        chunk = message[i : i + l_size]                                  # STEP 5.: Divide our message into blocks of 32 bytes
        
        digest = hashes.Hash(hashes.SHA256())              
                                                                         # STEP 6.: Hash each block, measuring the hashing time
        hash_update_time = measure_performance_time(digest.update, block_index.to_bytes(4, 'big') + r_bytes)[0]
        hash_finalize_time, block_with_hash = measure_performance_time(digest.finalize)
        hash_time += hash_update_time + hash_finalize_time
        
        ciphertext_blocks.append(xor_bytes(chunk, block_with_hash))      # STEP 7.: Apply XOR between original block and hashed block, and append to ciphertext

        return b"".join(ciphertext_blocks), hash_time                    # Return the full ciphertext and the hashing time, for future performance analysis

Now, we simply need to generate the private key and its corresponding public key, to perform encryption and decryption.

In [66]:
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key() 

Which leaves us ready to encrypt our files using that same public key. We start by using the function get_files (available in file aux_func.py), which allows us to choose a file size to test our RSA function on, and then returns 10 files of that size and the path to the corresponding parent folder.

In [67]:
base_dir = os.getcwd()

def main(mode):
    all_files, folder_path, file_size = get_files(base_dir, mode)              # Get 10 files of a chosen size, the path of their parent folder and that same chosen size.
    total_time = 0

    print(f"\n{mode} {len(all_files)} files from folder '{file_size}'...")
    print(f"{'File':<20} | {'Size':<10} | {'Hash Time (s)':<12} | {'RSA Time (s)':<12}")
    print("-" * 65)
    if (all_files):
        for file_name in all_files:                                      # For each of the files, encrypt.
            file_path = os.path.join(folder_path, file_name)

            with open(file_path, "rb") as f: 
                data = f.read()
                                                                     # Measure the time of encryption for future performance analysis.
            rsa_time, result = measure_performance_time(custom_encrypt_decrypt, [private_key, public_key, data, mode])
            ciphertext, hash_time = result
            total_time += rsa_time                                                      # Save the encrypted files in a new folder.        
            output_dir = save_in_results(ciphertext, base_dir, file_size, file_name, mode)
            print(f"{file_name:<20} | {len(data):>10} | {hash_time:12.6f} | {rsa_time:12.6f}")
            print("-" * 65)
        
        print(f"Total time for folder {file_size}: {total_time:.6f} seconds")
        print(f"Files saved in: {output_dir}\n")
    else: print("The folder is empty.")
    return output_dir

main("Encrypting")

TypeError: get_files() takes 1 positional argument but 2 were given

In [ ]:
output_dir = main("Decrypting")
print(output_dir)

With these results, we can now make a thorough analysis of the performance of our program. Let's start by comparing the execution times of our RSA-based encryption for each of the file sizes.